# Punto 4 — Limpieza de la columna de teléfono

**Curso:** IA en Producción — UDLA
**Alumno:** Pablo Villegas
**Rama:** `Villegates`
**Fuente:** `plan_de_compras_2025.xlsx`

**Actividad asignada (punto 4):**
> Limpieza de columnas de teléfono, colocar anexo en columna separada y limpieza de los
> teléfonos (quitar espacios y guiones).

Qué hace este notebook:

1. Carga el archivo Excel `plan_de_compras_2025.xlsx`.
2. Diagnostica los formatos con que viene escrita la columna `Teléfono responsable`.
3. Define la regla para reconocer el anexo dentro del texto original.
4. Crea dos columnas nuevas: `Teléfono limpio` (solo dígitos) y `Anexo`.
5. Compara el antes y el después de cada formato encontrado.
6. Valida el resultado (largos, casos sin dato, cantidad de anexos).

**Requisitos:** `pandas` y `openpyxl` (`pip install pandas openpyxl`).

## 1. Cargar el archivo Excel

In [1]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# El Excel está en la raíz del repositorio; este notebook vive en notebooks/.
# Se buscan ambas ubicaciones para que funcione en cualquiera de las dos.
ARCHIVO = "plan_de_compras_2025.xlsx"
ruta = next(p for p in (Path(ARCHIVO), Path("..") / ARCHIVO) if p.exists())

df = pd.read_excel(ruta)
COL_TEL = "Teléfono responsable"

print(f"Archivo leído: {ruta}")
print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
print(f"Columna a limpiar: {COL_TEL!r}")

Archivo leído: ..\plan_de_compras_2025.xlsx
Filas: 831  |  Columnas: 26
Columna a limpiar: 'Teléfono responsable'


## 2. Diagnóstico de la columna `Teléfono responsable`

Antes de limpiar hay que ver **cómo viene escrita** la columna: si tiene nulos, espacios,
guiones sueltos al final o un anexo pegado al número.

In [2]:
tel_original = df[COL_TEL].astype(str).str.strip()

print(f"Registros totales:        {len(df)}")
print(f"Valores nulos:            {df[COL_TEL].isna().sum()}")
print(f"Valores distintos:        {tel_original.nunique()}")
print(f"Contienen espacios:       {tel_original.str.contains(' ').sum()}")
print(f"Contienen guiones:        {tel_original.str.contains('-').sum()}")
print(f"Contienen letras:         {tel_original.str.contains('[A-Za-z]').sum()}")
print()
print("Cantidad de guiones por valor:")
print(tel_original.str.count("-").value_counts().sort_index().to_string())

Registros totales:        831
Valores nulos:            0
Valores distintos:        96
Contienen espacios:       43
Contienen guiones:        764
Contienen letras:         0

Cantidad de guiones por valor:
Teléfono responsable
0     67
2    764


In [3]:
# Los 15 formatos más frecuentes tal como vienen en el Excel
tel_original.value_counts().head(15)

Teléfono responsable
02-29011155-       325
22-9011135-         40
29-011468-          39
29-011201-          38
22-9011395-         37
29-011170-          26
22-9011325-1325     23
52-2432228-         22
02-29011325-        19
0                   14
512699924           14
72-2 350701-        14
32--2186811         14
322186802            9
2210208              8
Name: count, dtype: int64

## 3. Regla para separar el anexo

Mirando los datos aparecen cuatro situaciones distintas:

| Valor original | Qué es |
|---|---|
| `322186802` | teléfono limpio, sin anexo |
| `02-29011170-` | teléfono con guiones y un guion suelto al final (sin anexo) |
| `2 -2901 1198-` | teléfono con espacios *dentro* del número (sin anexo) |
| `20-9011468-1468` | teléfono **y** anexo al final |

La regla que se aplica es:

1. Se quitan los guiones y espacios sobrantes del final.
2. Si el texto termina en `-` seguido de **1 a 5 dígitos**, ese bloque es candidato a anexo.
3. Solo se acepta como anexo si **lo que queda delante sigue teniendo al menos 7 dígitos**
   (es decir, sigue siendo un teléfono completo). Esto evita cortar mal casos como
   `29-011141-`, donde `01141` parece anexo pero delante solo quedaría `29`.
4. Al teléfono resultante se le quita **todo lo que no sea dígito** (espacios, guiones, etc.).
5. Valores basura como `0` quedan como nulo, no se inventa un número.

In [4]:
SOLO_DIGITOS = re.compile(r"\D")                              # todo lo que no sea dígito
PATRON_ANEXO = re.compile(r"^(?P<base>.*)-\s*(?P<anexo>\d{1,5})\s*$")

LARGO_MIN_TELEFONO = 7   # delante del anexo debe quedar un teléfono completo
LARGO_MIN_VALIDO = 4     # menos que esto no es un teléfono (ej.: "0")


def separar_telefono_y_anexo(valor):
    """Devuelve (teléfono solo dígitos, anexo) a partir del texto original."""
    if pd.isna(valor):
        return pd.NA, pd.NA

    texto = str(valor).strip().rstrip("- ")   # 3.1 quita guiones/espacios sueltos del final
    anexo = pd.NA

    coincidencia = PATRON_ANEXO.match(texto)  # 3.2 ¿termina en "-" + 1 a 5 dígitos?
    if coincidencia:
        base = coincidencia.group("base")
        if len(SOLO_DIGITOS.sub("", base)) >= LARGO_MIN_TELEFONO:   # 3.3
            texto = base
            anexo = coincidencia.group("anexo")

    telefono = SOLO_DIGITOS.sub("", texto)    # 3.4 quita espacios, guiones y otros signos
    if len(telefono) < LARGO_MIN_VALIDO:      # 3.5
        return pd.NA, anexo

    return telefono, anexo

In [5]:
# Prueba de la función con un caso de cada tipo encontrado en el archivo
casos = [
    "322186802",          # ya viene limpio
    "32-2186802-",        # guiones + guion suelto al final
    "2 -2901 1198-",      # espacios dentro del número
    "72-2 350701-",       # espacios y guiones mezclados
    "20-9011468-1468",    # teléfono + anexo
    "22-901 1325-1325",   # teléfono con espacios + anexo
    "29-01141-",          # NO es anexo: delante quedarían solo 2 dígitos
    "32--2186811",        # guion doble en medio
    "0",                  # sin dato real
]

pd.DataFrame(
    [(c, *separar_telefono_y_anexo(c)) for c in casos],
    columns=["Original", "Teléfono limpio", "Anexo"],
)

,Original,Teléfono limpio,Anexo
0,322186802,322186802,NaN
1,32-2186802-,322186802,NaN
2,2 -2901 1198-,229011198,NaN
3,72-2 350701-,722350701,NaN
4,20-9011468-1468,209011468,1468
5,22-901 1325-1325,229011325,1325
6,29-01141-,2901141,NaN
7,32--2186811,322186811,NaN
8,0,NaN,NaN


## 4. Aplicar la limpieza y crear las columnas

`Teléfono limpio` y `Anexo` se insertan **al lado** de la columna original, para poder
comparar sin perder el dato de origen.

In [6]:
limpio = df[COL_TEL].apply(separar_telefono_y_anexo)

df["Teléfono limpio"] = [t for t, _ in limpio]
df["Anexo"] = [a for _, a in limpio]

# Reordenar: dejar las columnas nuevas justo después del teléfono original
orden = list(df.columns)
for nueva in ("Teléfono limpio", "Anexo"):
    orden.remove(nueva)
pos = orden.index(COL_TEL) + 1
orden[pos:pos] = ["Teléfono limpio", "Anexo"]
df = df[orden]

df[["Nombre responsable", COL_TEL, "Teléfono limpio", "Anexo"]].head(10)

,Nombre responsable,Teléfono responsable,Teléfono limpio,Anexo
0,María Teresa Zúñiga Silva,2 -2901 1198-,229011198,NaN
1,Ariel Gardaix Gardaix,32-2186802-,322186802,NaN
2,Ariel Gardaix Gardaix,32-2186802-,322186802,NaN
3,Ariel Gardaix Gardaix,32-2186802-,322186802,NaN
4,Ariel Gardaix Gardaix,322186802,322186802,NaN
5,Sebastian Cerda Peña,02-29011184-,0229011184,NaN
6,Victor Silva Aguilar,22-901727-,22901727,NaN
7,Jovita Gavilan González,0,NaN,NaN
8,Jovita Gavilan González,0,NaN,NaN
9,Victor Silva Aguilar,22901727,22901727,NaN


## 5. Antes y después de cada formato

In [7]:
comparacion = (
    df[[COL_TEL, "Teléfono limpio", "Anexo"]]
    .astype({COL_TEL: str})
    .drop_duplicates()
    .rename(columns={COL_TEL: "Original"})
    .sort_values("Original")
    .reset_index(drop=True)
)

print(f"Formatos distintos en el archivo: {len(comparacion)}")
comparacion.head(30)

Formatos distintos en el archivo: 96


,Original,Teléfono limpio,Anexo
0,0,NaN,NaN
1,02-29011082-,0229011082,NaN
2,02-290111082-,02290111082,NaN
3,02-290111325-,02290111325,NaN
4,02-29011155-,0229011155,NaN
5,02-29011170-,0229011170,NaN
6,02-29011184-,0229011184,NaN
7,02-29011248-,0229011248,NaN
8,02-29011325-,0229011325,NaN
9,02-29011385-,0229011385,NaN


In [8]:
# Solo los registros donde efectivamente se separó un anexo
con_anexo = comparacion[comparacion["Anexo"].notna()]
print(f"Formatos con anexo: {len(con_anexo)}")
con_anexo

Formatos con anexo: 28


,Original,Teléfono limpio,Anexo
11,2 -2901 1170-1170,229011170,1170
13,2 -2901 1273-1273,229011273,1273
14,2 -2901 1325-1325,229011325,1325
15,2 -2901 1359-1359,229011359,1359
16,2 -2901 1395-1395,229011395,1395
18,2-29011325-1325,229011325,1325
19,2-9011325-1325,29011325,1325
20,20-9011468-1468,209011468,1468
21,22-29011727-1727,2229011727,1727
23,22-901 1170-1170,229011170,1170


## 6. Validación del resultado

Tres chequeos: que no queden espacios ni guiones, cuántos teléfonos quedaron sin dato y
qué largos tienen los números resultantes.

In [9]:
solo_digitos = df["Teléfono limpio"].dropna().str.fullmatch(r"\d+").all()
anexo_ok = df["Anexo"].dropna().str.fullmatch(r"\d+").all()

print(f"Todos los teléfonos son solo dígitos: {solo_digitos}")
print(f"Todos los anexos son solo dígitos:    {anexo_ok}")
print()
print(f"Teléfonos con dato:      {df['Teléfono limpio'].notna().sum()}")
print(f"Teléfonos sin dato (NA): {df['Teléfono limpio'].isna().sum()}")
print(f"Registros con anexo:     {df['Anexo'].notna().sum()}")
print(f"Anexos distintos:        {df['Anexo'].nunique()}")

Todos los teléfonos son solo dígitos: True
Todos los anexos son solo dígitos:    True

Teléfonos con dato:      817
Teléfonos sin dato (NA): 14
Registros con anexo:     70
Anexos distintos:        17


In [10]:
# Largo de los teléfonos ya limpios (en Chile lo esperable es 8 o 9 dígitos)
largos = df["Teléfono limpio"].dropna().str.len().value_counts().sort_index()
largos.rename("Cantidad").rename_axis("Largo").to_frame()

,Cantidad
Largo,
7,17
8,136
9,299
10,355
11,10


In [11]:
# Los que quedan fuera de 8-9 dígitos son datos mal cargados en el origen:
# se dejan visibles en vez de corregirlos a mano.
fuera_de_rango = df["Teléfono limpio"].dropna()
fuera_de_rango = fuera_de_rango[~fuera_de_rango.str.len().between(8, 9)]

(
    df.loc[fuera_de_rango.index, [COL_TEL, "Teléfono limpio", "Anexo"]]
    .astype({COL_TEL: str})
    .drop_duplicates()
    .rename(columns={COL_TEL: "Original"})
)

,Original,Teléfono limpio,Anexo
5,02-29011184-,0229011184,NaN
66,2210208,2210208,NaN
163,02-29011325-,0229011325,NaN
183,02-29011170-,0229011170,NaN
213,90-11370-,9011370,NaN
255,29-01240-,2901240,NaN
257,29-01141-,2901141,NaN
258,56-912345678-,56912345678,NaN
273,29-01223-,2901223,NaN
302,29-01148-,2901148,NaN


## 7. Resumen

La columna `Teléfono responsable` quedó separada en dos columnas limpias:

- **`Teléfono limpio`**: solo dígitos, sin espacios ni guiones.
- **`Anexo`**: el anexo en su propia columna, o nulo cuando el registro no lo trae.

La columna original se conserva para poder auditar la transformación.

In [12]:
resumen = pd.DataFrame(
    {
        "Métrica": [
            "Registros totales",
            "Formatos distintos antes",
            "Teléfonos distintos después",
            "Registros con anexo",
            "Registros sin teléfono válido",
        ],
        "Valor": [
            len(df),
            df[COL_TEL].astype(str).nunique(),
            df["Teléfono limpio"].nunique(),
            int(df["Anexo"].notna().sum()),
            int(df["Teléfono limpio"].isna().sum()),
        ],
    }
)
resumen

,Métrica,Valor
0,Registros totales,831
1,Formatos distintos antes,96
2,Teléfonos distintos después,74
3,Registros con anexo,70
4,Registros sin teléfono válido,14


In [13]:
df[["Unidad de Compra", "Nombre responsable", COL_TEL, "Teléfono limpio", "Anexo"]].head(15)

,Unidad de Compra,Nombre responsable,Teléfono responsable,Teléfono limpio,Anexo
0,Ministerio de Vivienda y Urbanismo(766),María Teresa Zúñiga Silva,2 -2901 1198-,229011198,NaN
1,SEREMI MINVU V REGION,Ariel Gardaix Gardaix,32-2186802-,322186802,NaN
2,SEREMI MINVU V REGION,Ariel Gardaix Gardaix,32-2186802-,322186802,NaN
3,SEREMI MINVU V REGION,Ariel Gardaix Gardaix,32-2186802-,322186802,NaN
4,SEREMI MINVU V REGION,Ariel Gardaix Gardaix,322186802,322186802,NaN
5,Ministerio de Vivienda y Urbanismo(766),Sebastian Cerda Peña,02-29011184-,0229011184,NaN
6,Ministerio de Vivienda y Urbanismo(766),Victor Silva Aguilar,22-901727-,22901727,NaN
7,Ministerio de Vivienda y Urbanismo(766),Jovita Gavilan González,0,NaN,NaN
8,Ministerio de Vivienda y Urbanismo(766),Jovita Gavilan González,0,NaN,NaN
9,Ministerio de Vivienda y Urbanismo(766),Victor Silva Aguilar,22901727,22901727,NaN
